### Convert your cleaned dataset into structured documents for RAG
Recipe: Tomato Pasta,
Category: meals,
Ingredients: tomato, garlic, pasta,
Instructions:
Boil pasta...

In [9]:
!pip install pandas
import pandas as pd
import ast

# Load cleaned dataset
df = pd.read_csv("recipes_cleaned.csv")

# Convert NER string → list
def parse_ner(ner):
    try:
        return ast.literal_eval(ner)
    except:
        return []

# Create structured document
def create_document(row):
    ingredients = ", ".join(parse_ner(row['NER']))
    
    return f"""
Recipe: {row['title']}
Category: {row['genre']}
Ingredients: {ingredients}

Instructions:
{row['directions']}
"""

# Apply transformation
df['document'] = df.apply(create_document, axis=1)

# Save new dataset
df.to_csv("recipes_with_docs.csv", index=False)

print("Documents created and saved!")

  Using cached pandas-3.0.2-cp314-cp314-win_amd64.whl.metadata (19 kB)
  Using cached numpy-2.4.4-cp314-cp314-win_amd64.whl.metadata (6.6 kB)
  Using cached tzdata-2026.1-py2.py3-none-any.whl.metadata (1.4 kB)
Using cached pandas-3.0.2-cp314-cp314-win_amd64.whl (9.9 MB)
Using cached numpy-2.4.4-cp314-cp314-win_amd64.whl (12.4 MB)
Using cached tzdata-2026.1-py2.py3-none-any.whl (348 kB)

   ---------------------------------------- 0/3 [tzdata]
   ---------------------------------------- 0/3 [tzdata]
   ---------------------------------------- 0/3 [tzdata]
   ---------------------------------------- 0/3 [tzdata]
   ------------- -------------------------- 1/3 [numpy]
   ------------- -------------------------- 1/3 [numpy]
   ------------- -------------------------- 1/3 [numpy]
   ------------- -------------------------- 1/3 [numpy]
   ------------- -------------------------- 1/3 [numpy]
   ------------- -------------------------- 1/3 [numpy]
   ------------- -------------------------- 1/

### Convert your documents into embeddings and store them in FAISS (Vector DB)

User query → vector → similarity search → relevant recipes

In [20]:
import pandas as pd
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_core.documents import Document

# Load dataset
df = pd.read_csv("recipes_with_docs.csv")

# Convert to LangChain documents
documents = [
    Document(page_content=row['document'])
    for _, row in df.iterrows()
]

# Load embedding model
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

# Create FAISS vector store
vector_db = FAISS.from_documents(documents, embeddings)

# Save locally
vector_db.save_local("recipe_faiss_index")

print("FAISS index created and saved!")

C:\Users\NarenkarthikSivaprab\AppData\Local\Temp\ipykernel_15560\2910279464.py:16: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(
c:\Users\NarenkarthikSivaprab\Desktop\jman projects\ai_mini\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\NarenkarthikSivaprab\Desktop\jman projects\ai_mini\.venv\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files bu

FAISS index created and saved!
